# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features 
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Imports

In [ ]:
#Imports
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

#import joypy
from scipy import stats

from plate_preprocessing import *


## Get table info for a single plate

In [ ]:
root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v5_active/"
filename = "output.db"

db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

plate = "20250501_rep07"
combine_dfs = True
calculate_medians = False

plate_dfs = {}

pre_pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
# metadata extraction
map_file = os.path.join("plate_metadata", f"{plate}_metadata", f"{plate}_map.csv")
if os.path.exists(map_file):
    print(map_file)
else:
    ValueError(f"Error: Plate metadata at {map_file} for plate {plate} not found")

pre_cell_df = pre_pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
print(f"Shape after merging image: {pre_cell_df.shape} from {pre_pre_cell_df.shape}")
# simplify the metadata column labels
pre_cell_df.columns = pre_cell_df.columns.str.replace(
    r"^Image_Metadata_", "Metadata_", regex=True
)
metadata_cols = [col for col in pre_cell_df.columns if "Metadata" in col]
# display(cell_df)

# merge the 1:1 dfs together
if combine_dfs:
    cell_df = combine_one_to_one_dfs(pre_cell_df, conn)
    
print(f"Shape after combining df: {cell_df.shape}")
# Remove null/infinite rows
cols_to_check = ["Metadata_WellRow", "Metadata_WellColumn", "Metadata_Field"]
cell_df = cell_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN
cell_df = cell_df.dropna(subset=cols_to_check)  # Drop rows with NaN in these columns
# display(cell_df)
print(f"Shape after dropping na: {cell_df.shape}")


# add the median data
if calculate_medians:
    extra_feature_filename = "extra_features.db"
    extra_feature_db_path = os.path.join(root, extra_feature_filename)
    cell_df = load_organelle_medians(db_path=extra_feature_db_path, df=cell_df)
    cell_df.reset_index(drop=True)

# Find cell/nuc area ratio
cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(
    cell_df, "Cell_AreaShape_Area", "Nuclei_AreaShape_Area"
)

# get rid of cells where area is lower than nuc area
#cell_df = cell_df[cell_df["Cell_Nuclei_Area_Ratio"] > 1]
# cell_df["Cell_Nuclei_Area_Ratio"].plot(kind="density",xlim=(-1,200))

display(
    cell_df[
        [
            "Cell_AreaShape_Area",
            "Nuclei_AreaShape_Area",
            "Cell_Nuclei_Area_Ratio",
            "Cell_Mean_Mitochondria_AreaShape_Area",
            #"Cell_Median_Mitochondria_AreaShape_Area",
        ]
    ]
)

# make these metadatas string
cell_df["Metadata_WellRow"] = cell_df["Metadata_WellRow"].astype(int)
cell_df["Metadata_WellColumn"] = cell_df["Metadata_WellColumn"].astype(int)
cell_df["Metadata_Field"] = cell_df["Metadata_Field"].astype(int)
display(cell_df[metadata_cols].head())


print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    #display(platemap_df)

    platemap_df["Metadata_WellRow"] = platemap_df["Metadata_WellRow"].astype(int)
    platemap_df["Metadata_WellColumn"] = platemap_df["Metadata_WellColumn"].astype(int)
    platemap_df["Metadata_Field"] = platemap_df["Metadata_Field"].astype(int)
    # platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(
        platemap_df,
        on=["Metadata_WellRow", "Metadata_WellColumn", "Metadata_Field","Metadata_Well"],
        how="left",
    )
    # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["PassageGroup"] = cell_df["PassageNumber"].apply(passage_group)
    cell_df["AllGroups"] = take_drug_from_condition(cell_df, "PassageGroup", "Drug","Doxo")
    
    cell_df["PlateNumber"] = cell_df["PlateNumber"].astype(int)
    
    # display(cell_df[cell_df["Metadata_Well"] == "A02"])
    # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])

min_x = 0
min_y = 0
max_x = cell_df["Image_Width_DAPI"][0]  # get the max x and y resolutions
max_y = cell_df["Image_Height_DAPI"][0]

# cell_df_excluded_borders = exclude_borders(
#     cell_df, min_x, min_y, max_x, max_y, prefix="Cell_"
# )
print(f"Shape after almost everything: {cell_df.shape}")
plate_dfs[plate] = cell_df

combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
print(f"Shape after everything: {combined_cell_df.shape}")
# display(combined_cell_df)

plate_metadata/20250501_rep07_metadata/20250501_rep07_map.csv
Shape after merging image: (10621, 1203) from (10621, 916)
Shape after combining df: (10621, 2761)
Shape after dropping na: (10621, 2761)


,Cell_AreaShape_Area,Nuclei_AreaShape_Area,Cell_Nuclei_Area_Ratio,Cell_Mean_Mitochondria_AreaShape_Area
0,618820.0,84863.0,7.291988,1881.182540
1,255888.0,64517.0,3.966210,576.141509
2,200440.0,31901.0,6.283189,2497.833333
3,18098.0,11990.0,1.509425,508.916667
4,185510.0,29852.0,6.214324,1820.893617
...,...,...,...,...
10616,116041.0,29558.0,3.925875,1228.600000
10617,61787.0,16298.0,3.791079,8344.333333
10618,76594.0,20589.0,3.720142,912.962963
10619,68268.0,29105.0,2.345576,2195.333333


,Image_ExecutionTime_02Metadata,Metadata_Channels,Metadata_EmptyImage_Cells,Metadata_EmptyImage_Nuclei,Metadata_Field,Metadata_FileLocation,Metadata_Frame,Metadata_Series,Metadata_Well,Metadata_WellColumn,Metadata_WellRow
0,0.0,None,0,0,1,None,0,0,C01,1,3
1,0.0,None,0,0,1,None,0,0,C01,1,3
2,0.0,None,0,0,12,None,0,0,C01,1,3
3,0.0,None,0,0,12,None,0,0,C01,1,3
4,0.0,None,0,0,9,None,0,0,C01,1,3


True
Shape after almost everything: (10621, 2776)
Shape after everything: (10621, 2776)


In [4]:
display(combined_cell_df)

,ImageNumber,Cell_Number_Object_Number,Cell_AreaShape_Area,Cell_AreaShape_BoundingBoxArea,Cell_AreaShape_BoundingBoxMaximum_X,Cell_AreaShape_BoundingBoxMaximum_Y,Cell_AreaShape_BoundingBoxMinimum_X,Cell_AreaShape_BoundingBoxMinimum_Y,Cell_AreaShape_Center_X,Cell_AreaShape_Center_Y,...,PassageNumber,Lineage,AgeGroup,Drug,FlaggedBatch,LineageNumber,TimepointName,Plate_Number,PassageGroup,AllGroups
0,1,1,618820.0,1316978.0,2160.0,1369.0,1198.0,0.0,1754.383452,530.672456,...,19,Doxo_LIN12-0C-b2-s2,10,Doxo,False,12,Doxo_P19 LIN12-0C-b2-s2 AG10,7,P19-21,Doxo
1,1,2,255888.0,408960.0,1117.0,2160.0,52.0,1776.0,632.865511,1974.368861,...,19,Doxo_LIN12-0C-b2-s2,10,Doxo,False,12,Doxo_P19 LIN12-0C-b2-s2 AG10,7,P19-21,Doxo
2,12,1,200440.0,489684.0,1301.0,516.0,352.0,0.0,875.641504,170.437263,...,19,Doxo_LIN12-0C-b2-s2,10,Doxo,False,12,Doxo_P19 LIN12-0C-b2-s2 AG10,7,P19-21,Doxo
3,12,2,18098.0,30132.0,432.0,684.0,270.0,498.0,349.955851,591.964471,...,19,Doxo_LIN12-0C-b2-s2,10,Doxo,False,12,Doxo_P19 LIN12-0C-b2-s2 AG10,7,P19-21,Doxo
4,9,1,185510.0,643344.0,1031.0,1866.0,0.0,1242.0,328.194076,1646.367301,...,19,Doxo_LIN12-0C-b2-s2,10,Doxo,False,12,Doxo_P19 LIN12-0C-b2-s2 AG10,7,P19-21,Doxo
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10616,2199,10,116041.0,218196.0,396.0,1858.0,0.0,1307.0,163.264062,1602.058884,...,13,LIN11-0C-b5-s1,0,NaN,False,11,P13 LIN11-0C-b5-s1 AG0,7,P13-15,P13-15
10617,2199,11,61787.0,111360.0,1245.0,1727.0,955.0,1343.0,1113.066875,1522.839740,...,13,LIN11-0C-b5-s1,0,NaN,False,11,P13 LIN11-0C-b5-s1 AG0,7,P13-15,P13-15
10618,2199,12,76594.0,203840.0,2160.0,2026.0,1744.0,1536.0,2006.284500,1787.241063,...,13,LIN11-0C-b5-s1,0,NaN,False,11,P13 LIN11-0C-b5-s1 AG0,7,P13-15,P13-15
10619,2199,13,68268.0,84410.0,777.0,1806.0,410.0,1576.0,582.425880,1686.259302,...,13,LIN11-0C-b5-s1,0,NaN,False,11,P13 LIN11-0C-b5-s1 AG0,7,P13-15,P13-15


# Exporting zone: 
Make sure you put "_active" in the CP output folder names!

In [6]:
# Setting file paths
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"
parent_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output"

parent_object_table = "Per_Cell"
compartment_tables = ["Nuclei","Cytoplasm"] #note that these must be in a 1:1 relationship with cell
total_organelles_tables = ["Per_TotalMitochondria", "Per_TotalLysosomes"]

# Initialize a list to store the combined DataFrames
plate_dfs = {}

# [EDIT] don't use this if using total organelles tables!!!
# query designed to remove cells from the dataset without any mitochondria, nuclei or lysosomes; allows us to have a 1:1 relationship
parent_obj_query = f"SELECT * FROM Per_{parent_object_table} WHERE Cell_Children_Mitochondria_Count > 0 AND Cell_Children_Lysosomes_Count > 0 AND Cell_Children_Nuclei_Count > 0;"

In [7]:
def load_and_combine_plates_from_db(
    parent_dir,
    curr_plates,
    main_compartment_name="Cell",
    combine_dfs=True,
    combine_total_organelles=True,
    total_organelles_tables=None,
    calculate_medians=True,
):
    # Loop over the plates
    for root, dirs, files in os.walk(parent_dir):
        for filename in files:
            if (
                filename.endswith(".db")
                and "active" in root
                and "extra" not in filename
                and "output" in filename
            ):
                db_path = os.path.join(root, filename)  # make the path
                for plate in curr_plates:
                    if plate in db_path:
                        conn = sqlite3.connect(db_path)
                        cursor = conn.cursor()
                        # update_database_with_well_metadata(db_path)
                        try:
                            # Read the 'Per_Cell' table and get metadata from 'Per_Image' table
                            pre_cell_df = pd.read_sql_query(
                                f"SELECT * FROM Per_{main_compartment_name}", conn
                            )
                            image_df = pd.read_sql_query(
                                "SELECT * FROM Per_Image", conn
                            )
                            # metadata extraction - make sure its the exact same file format as the one above
                            map_file = os.path.join(
                                "plate_metadata", f"{plate}_metadata", f"{plate}_map.csv"
                            )
                            if os.path.exists(map_file):
                                print(map_file)
                            else:
                                raise ValueError(f"Error: Plate metadata at {map_file} for plate {plate} not found")

                            cell_df = pre_cell_df.merge(
                                image_df, on=["ImageNumber"], how="left"
                            )
                            cell_df.columns = cell_df.columns.str.replace(
                                r"^Image_Metadata_", "Metadata_", regex=True
                            )
                            # merge dfs together that are 1:1 e.g nuc and cyto with cell
                            if combine_dfs:
                                cell_df = combine_one_to_one_dfs(cell_df, conn, tables_to_add=compartment_tables)

                            # remove rows where there isn't a valid row/column/field metadata
                            cols_to_check = [
                                "Metadata_WellRow",
                                "Metadata_WellColumn",
                                "Metadata_Field",
                            ]
                            cell_df = cell_df.replace(
                                [np.inf, -np.inf], np.nan
                            )  # Replace inf with NaN
                            cell_df = cell_df.dropna(
                                subset=cols_to_check
                            )  # Drop rows with NaN in these columns

                            # make these metadatas int
                            cell_df["Metadata_WellRow"] = cell_df[
                                "Metadata_WellRow"
                            ].astype(int)
                            cell_df["Metadata_WellColumn"] = cell_df[
                                "Metadata_WellColumn"
                            ].astype(int)
                            cell_df["Metadata_Field"] = cell_df[
                                "Metadata_Field"
                            ].astype(int)

                            # add the median data
                            if calculate_medians:
                                extra_feature_filename = "extra_features.db"
                                extra_feature_db_path = os.path.join(
                                    root, extra_feature_filename
                                )
                                cell_df = load_organelle_stats(
                                    db_path=extra_feature_db_path, df=cell_df
                                )
                                cell_df = cell_df.reset_index(drop=True)

                            # Find cell/nuc area ratio
                            cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(
                                cell_df,
                                cell_area_col="Cell_AreaShape_Area",
                                nuc_area_col="Nuclei_AreaShape_Area",
                            )
                            
                            if combine_total_organelles:
                                if total_organelles_tables is None:
                                    total_organelles_tables = ["Per_TotalMitochondria", "Per_TotalLysosomes"]
                                # Merge total mitochondria and lysosome data into the cell dataframe
                                total_mito_df = pd.read_sql_query(
                                    f"SELECT * FROM {total_organelles_tables[0]}", conn
                                )
                                total_lyso_df = pd.read_sql_query(
                                    f"SELECT * FROM {total_organelles_tables[1]}", conn
                                )
                                cell_df = merge_totalobject_df_into_parent_df(
                                    cell_df, total_mito_df, key_col="ImageNumber_Object_Number", rename_key=True, verbose=True
                                )
                                cell_df = merge_totalobject_df_into_parent_df(
                                    cell_df, total_lyso_df, key_col="ImageNumber_Object_Number", rename_key=True, verbose=False
                                )
                            # # get rid of cells where area is lower than nuc area
                            # cell_df = cell_df[
                            #     cell_df["Cell_Nuclei_Area_Ratio"] > 1
                            # ].reset_index(drop=True)
                            #Add helpful metadata columns for downstream analysis
                            
                            if os.path.exists(map_file):
                                platemap_df = pd.read_csv(map_file)
                                #display(platemap_df)
                                platemap_df["Metadata_WellRow"] = platemap_df[
                                    "Metadata_WellRow"
                                ].astype(int)
                                platemap_df["Metadata_WellColumn"] = platemap_df[
                                    "Metadata_WellColumn"
                                ].astype(int)
                                platemap_df["Metadata_Field"] = platemap_df[
                                    "Metadata_Field"
                                ].astype(int)

                                # platemap_df.reset_index(drop=True)
                                cell_df = cell_df.merge(
                                    platemap_df,
                                    on=[
                                        "Metadata_WellRow",
                                        "Metadata_WellColumn",
                                        "Metadata_Field",
                                        "Metadata_Well"
                                    ],
                                    how="left",
                                )
                                # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
                                cell_df["Passage Group"] = cell_df[
                                    "PassageNumber"
                                ].apply(passage_group)
                                cell_df["AllGroups"] = add_drug_to_group(
                                    cell_df, "Passage Group", "Drug"
                                )
                            

                            plate_dfs[plate] = cell_df
                        except Exception as e:
                            print(f"Error reading {db_path}: {e}")
                        finally:
                            conn.close()

    # Combine all DataFrames
    combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
    return combined_cell_df


In [8]:
combined_cell_df = load_and_combine_plates_from_db(parent_dir,curr_plates)
#combined_nuclei_df = pd.concat(nuclei_dfs.values(), ignore_index=True)
                
#Filter DataFrames to only include cells that were stained with LAMP1-488 and MitoRed



plate_metadata/20250410_rep06_metadata/20250410_rep06_map.csv
Number of objects in dropped DataFrame for Image 1289: 4
Number of objects in dropped DataFrame for Image 2109: 1
Number of objects in dropped DataFrame for Image 2398: 8
Number of objects in dropped DataFrame for Image 2489: 3
dropped_parent_df shape: (14574, 2880), object_df shape: (14574, 61), original parent_df shape: (14582, 2878)
merged_parent_df shape: (14574, 2940)
plate_metadata/20240326_rep02_metadata/20240326_rep02_map.csv
dropped_parent_df shape: (9361, 2880), object_df shape: (9361, 61), original parent_df shape: (9361, 2878)
merged_parent_df shape: (9361, 2940)
plate_metadata/20250501_rep07_metadata/20250501_rep07_map.csv
Number of objects in dropped DataFrame for Image 1537: 1
dropped_parent_df shape: (10618, 2880), object_df shape: (10618, 61), original parent_df shape: (10621, 2878)
merged_parent_df shape: (10618, 2940)
plate_metadata/20241112_rep04_metadata/20241112_rep04_map.csv
Number of objects in droppe

## Export everything to CSV

In [9]:
#Export to a giant csv
# filter out the non-experimental test images
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]

display(combined_cell_df_mitolyso.head(10))
# print(cell_df.shape, " ", filter_df.shape)

#export the plate
outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
combined_cell_df_mitolyso.to_csv(
    os.path.join(outpath, "total_combined_cell.csv"), index=False
)


,ImageNumber,Cell_Number_Object_Number,Cell_AreaShape_Area,Cell_AreaShape_BoundingBoxArea,Cell_AreaShape_BoundingBoxMaximum_X,Cell_AreaShape_BoundingBoxMaximum_Y,Cell_AreaShape_BoundingBoxMinimum_X,Cell_AreaShape_BoundingBoxMinimum_Y,Cell_AreaShape_Center_X,Cell_AreaShape_Center_Y,...,PassageNumber,Lineage,AgeGroup,Drug,FlaggedBatch,LineageNumber,TimepointName,Plate_Number,Passage Group,AllGroups
0,1,1,164214.0,529720.0,1538.0,680.0,759.0,0.0,1214.520144,288.284890,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
1,1,2,104987.0,341880.0,1881.0,616.0,1326.0,0.0,1730.603799,328.968777,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
2,1,3,304312.0,667457.0,2086.0,1434.0,1407.0,451.0,1688.153264,1002.163921,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
3,1,4,60598.0,134460.0,2160.0,973.0,1836.0,558.0,2017.926763,766.248193,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
4,8,1,26066.0,36036.0,1274.0,252.0,1131.0,0.0,1190.996355,129.453196,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
5,8,2,68589.0,172710.0,1450.0,857.0,1165.0,251.0,1301.820511,555.102334,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
6,12,1,155542.0,403370.0,2074.0,2160.0,1688.0,1115.0,1888.754002,1563.320492,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
7,7,1,214073.0,698055.0,1177.0,2160.0,370.0,1295.0,879.191229,1608.467350,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
8,13,1,212926.0,912288.0,1886.0,1248.0,1155.0,0.0,1586.334952,424.895659,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo
9,15,1,110434.0,199004.0,2160.0,1890.0,1804.0,1331.0,2004.931416,1570.705281,...,18,Doxo_LIN13-0C-b2-s3,9,Doxo,False,13,Doxo_P18 LIN13-0C-b2-s3 AG9,6.0,P16-18,Doxo


In [10]:

#set up the image borders and make a csv with excluded border cells
min_x = 0
min_y = 0
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]

combined_cell_df_mitolyso_borders_excluded = exclude_borders(
    combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_"
)
combined_cell_df_mitolyso_borders_excluded.to_csv(
    os.path.join(outpath, "total_combined_cell_borders_excluded.csv"), index=False
)


In [11]:
display(combined_cell_df_mitolyso[["Cell_Number_Object_Number", "Nuclei_Number_Object_Number","Cell_Parent_Nuclei"]].head(10))


,Cell_Number_Object_Number,Nuclei_Number_Object_Number,Cell_Parent_Nuclei
0,1,1,1
1,2,2,2
2,3,4,4
3,4,3,3
4,1,1,1
5,2,2,2
6,1,1,1
7,1,1,1
8,1,1,1
9,1,1,1


### Summary Stats

In [ ]:
def passage_groups_sort_key(group_name):
    """
    Key function for natural sorting of strings containing numbers.
    Extract numeric parts and convert to int .
    """
    digit_pattern = r"([0-9]+)"  # Matches "RX" where X is the Plate number (placeholder for now)
    match = re.search(digit_pattern, group_name)
    if match:
        first_digit = int(match.group(1))
        return first_digit
    else:
        text = group_name.lower()
        if text == "doxo":
            return 999
        else:
            return ValueError


def make_summary_stats_for_df_and_feature(
    df,
    x_value,
    feature,
    summary_outpath,
    df_tag="original",
    plate_col_name="PlateNumber",
    feature_name="area",
    group_name="passage_group",
    include_cols=[],
):
    from pathlib import Path
    try:
        table_csvname = f"{df_tag}_total_combined_{feature_name}_stats.csv"
        feature_csvname = f"{df_tag}_{feature_name}_by_{group_name}_stats.csv"
        agg_feature_csvname = f"{df_tag}_agg_{feature_name}_by_{group_name}_stats.csv"

        subfolder_name = f"{df_tag}_{feature_name}_summary_stats"
        parent_folder = Path(summary_outpath, subfolder_name)
        parent_folder.mkdir(exist_ok=True)

        if not include_cols:
            df_to_summarize = df
        else:
            df_to_summarize = df[include_cols]
        df_to_summarize.describe().to_csv(
            os.path.join(summary_outpath, subfolder_name, table_csvname)
        )
        group_averages = df.groupby(
            [x_value, plate_col_name], as_index=False, observed=True
        )[feature]
        # Reset the index to get a clean DataFrame
        # average_df = group_averages.reset_index()
        avg_summary = group_averages.describe()
        avg_summary_sorted = avg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        avg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, feature_csvname)
        )

        # do the agg by passage group only
        group_averages_agg = df.groupby([x_value], as_index=False, observed=True)[
            feature
        ]
        avg_agg_summary = group_averages_agg.describe()
        avg_agg_summary_sorted = avg_agg_summary.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        )
        avg_agg_summary_sorted.to_csv(
            os.path.join(summary_outpath, subfolder_name, agg_feature_csvname)
        )
        print(
            f"saved files {(table_csvname, feature_csvname, agg_feature_csvname)} to {summary_outpath}"
        )
        return True
    except ValueError as e:
        print(f"Could not make summary stats: {e}")
        return False
    
# summary_outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/Cell_Size_Data/summary_stats/"

# for feature in ["Cell_AreaShape_Area","Nuclei_AreaShape_Area","Cell_Nuclei_Area_Ratio"]:
#     make_summary_stats_for_df_and_feature(
#         combined_cell_df_mitolyso,
#         "AllGroups",
#         feature,
#         summary_outpath,
#         df_tag="original",
#         feature_name=feature,
#         include_cols=[
#             "Cell_Number_Object_Number",
#             "Cell_AreaShape_Area",
#             "Nuclei_AreaShape_Area",
#             "Cell_Nuclei_Area_Ratio",
#             "Cell_Children_Lysosomes_Count",
#             "Cell_Children_Mitochondria_Count",
#         ],
#     )
#     make_summary_stats_for_df_and_feature(
#         combined_cell_df_mitolyso_borders_excluded,
#         "AllGroups",
#         feature,
#         summary_outpath,
#         df_tag="borders_excluded",
#         feature_name=feature,
#         include_cols=[
#             "Cell_Number_Object_Number",
#             "Cell_AreaShape_Area",
#             "Nuclei_AreaShape_Area",
#             "Cell_Nuclei_Area_Ratio",
#             "Cell_Children_Lysosomes_Count",
#             "Cell_Children_Mitochondria_Count",
#         ],
#     )
# display(combined_cell_df_mitolyso)